# Phase 3: Advanced Modeling (v2)

In this notebook, we'll use CatBoost, LightGBM, and XGBoost with Optuna tuning to maximize ROC-AUC.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import optuna
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
import warnings
warnings.filterwarnings('ignore')


In [2]:
X_train = np.load('../data/X_train_adv.npy')
X_test = np.load('../data/X_test_adv.npy')
y_train = np.load('../data/y_train_adv.npy')
y_test = np.load('../data/y_test_adv.npy')
with open('../data/feature_names_adv.pkl', 'rb') as f: feature_names = pickle.load(f)


In [3]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 5)
    }
    model = XGBClassifier(**params, random_state=42, use_label_encoder=False, eval_metric='logloss')
    model.fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

# study_xgb = optuna.create_study(direction='maximize')
# study_xgb.optimize(objective_xgb, n_trials=20) # Keeping trials low for speed


In [4]:
lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42)
cat = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_state=42, verbose=0)
xgb = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)

ensemble = VotingClassifier(estimators=[('lgb', lgb), ('cat', cat), ('xgb', xgb)], voting='soft')
ensemble.fit(X_train, y_train)

y_prob = ensemble.predict_proba(X_test)[:, 1]
print(f'Ensemble ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, ensemble.predict(X_test)))


[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001280 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1657
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265353 -> initscore=-1.018328
[LightGBM] [Info] Start training from score -1.018328


Ensemble ROC-AUC: 0.8338
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [5]:
with open('../models/advanced_ensemble.pkl', 'wb') as f: pickle.dump(ensemble, f)
